In [1]:
#---------------Definition of A3C---------------------

#A3C (Asynchronous Advantage Actor-Critic) is a Deep Reinforcement Learning algorithm
#that uses multiple agents (workers) to learn simultaneously
#and uses Actor-Critic networks to select actions and evaluate their performance.

In [2]:
%%writefile a3c.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
import gymnasium as gym

GAMMA,N_STEPS,LR,WORKERS,MAX_EP=0.99,5,1e-3,4,200

class Net(nn.Module):
    def __init__(self,s,a):
        super().__init__()
        self.body=nn.Sequential(nn.Linear(s,128),nn.ReLU())
        self.actor=nn.Linear(128,a)
        self.critic=nn.Linear(128,1)

    def forward(self,x):
        h=self.body(x)
        return self.actor(h),self.critic(h)

    def act(self,s):
        x=torch.tensor(s,dtype=torch.float32).unsqueeze(0)
        logits,v=self(x)
        d=torch.distributions.Categorical(F.softmax(logits,-1))
        a=d.sample()
        return a.item(),d.log_prob(a),v

def worker(wid,g,opt,c,q):
    env=gym.make("CartPole-v1")
    local=Net(4,2)
    local.load_state_dict(g.state_dict())

    while c.value<MAX_EP:
        s,_=env.reset()
        done=False
        ep_r=0
        lp,vals,rews=[],[],[]

        while not done:
            a,l,v=local.act(s)
            s,r,t,tr,_=env.step(a)
            done=t or tr
            lp.append(l);vals.append(v);rews.append(r)
            ep_r+=r

            if len(rews)==N_STEPS or done:
                R=0 if done else local.act(s)[2].item()
                ret=[]
                for r in rews[::-1]:
                    R=r+GAMMA*R
                    ret.insert(0,R)

                ret=torch.tensor(ret,dtype=torch.float32)
                adv=ret-torch.cat(vals).squeeze(-1)
                loss=-(torch.stack(lp)*adv.detach()).mean()+0.5*adv.pow(2).mean()

                opt.zero_grad()
                loss.backward()

                for x,y in zip(local.parameters(),g.parameters()):
                    y._grad=x.grad

                opt.step()
                local.load_state_dict(g.state_dict())
                lp,vals,rews=[],[],[]

        with c.get_lock():
            c.value+=1
            ep=c.value
        q.put((wid,ep,ep_r))

    env.close()

def main():
    g=Net(4,2)
    g.share_memory()
    opt=torch.optim.Adam(g.parameters(),lr=LR)
    c=mp.Value("i",0)
    q=mp.Queue()

    ps=[mp.Process(target=worker,args=(i,g,opt,c,q)) for i in range(WORKERS)]
    for p in ps:p.start()

    while True:
        wid,ep,r=q.get()
        if ep%10==0:
            print(f"ep={ep} worker={wid} reward={r}")
        if ep>=MAX_EP: break

    for p in ps:p.join()

if __name__=="__main__":
    mp.set_start_method("spawn",force=True)
    main()

Overwriting a3c.py


In [3]:
%env KMP_DUPLICATE_LIB_OK=TRUE
!python a3c.py

env: KMP_DUPLICATE_LIB_OK=TRUE
ep=10 worker=2 reward=35.0
ep=20 worker=2 reward=14.0
ep=30 worker=0 reward=12.0
ep=40 worker=0 reward=9.0
ep=50 worker=3 reward=15.0
ep=60 worker=2 reward=11.0
ep=70 worker=3 reward=34.0
ep=80 worker=3 reward=11.0
ep=90 worker=0 reward=10.0
ep=100 worker=3 reward=11.0
ep=110 worker=1 reward=10.0
ep=120 worker=0 reward=11.0
ep=130 worker=2 reward=10.0
ep=140 worker=1 reward=8.0
ep=150 worker=3 reward=9.0
ep=160 worker=1 reward=9.0
ep=170 worker=3 reward=10.0
ep=180 worker=0 reward=8.0
ep=190 worker=3 reward=10.0
ep=200 worker=2 reward=9.0
